In [1]:
import requests
from selenium import webdriver
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np

def get_page_html(url):
    driver = webdriver.Chrome()
    driver.get(url)
    html = driver.page_source
    driver.quit()
    soup = BeautifulSoup(html, 'html.parser')
    return soup

def extract_items(page):
    items = []
    who_s_listing = page.find('h1', class_=re.compile(r"^GiftGiverRegistryHeader-styles__Title__rTEe1"))
    if who_s_listing:
        who_s_listing = who_s_listing.get_text(strip=True)
    else:
        who_s_listing = "Unknown"

    for div in page.find_all('div',
        class_=re.compile(r"^RegistryCategory-styles__RegistryCategory")
        ):
        h2 = div.find('h2',
            class_=re.compile(r"^RegistryCategory-styles__Title")
        )
        if not h2:
            continue

        category = h2.get_text(strip=False)
        category = re.sub(r"\s*\(\d+\)$", "", category)
        
        for product in div.find_all(
            'div',
            class_=re.compile(r"^RegItemMinimalCard-styles__Content")
            ):
            product_name = product.find('div', 
                class_=re.compile(r"^RegItemMinimalCardDetails-styles__Title")
                ).get_text(strip=True)
            
            product_price = product.find('span', 
                class_=re.compile(r"^PricingTreatment-styles__PricingTreatment__Price")
                )
            if product_price:
                product_price = product_price.get_text(strip=True)
            else:
                product_price = "N/A"
            product_stores = [ store.get_text(strip=True).replace(" Icon", "") 
                for store in product.find_all('span',
                    class_=re.compile(r"^StoreIcons-styles")
                )
                if len(store.get_text(strip=True).replace(" Icon", "")) > 0
                ]
            product_stores.sort()
            items.append({
                "listing": who_s_listing,
                "category": category,
                "name": product_name,
                "price": product_price,
                "stores": ', '.join(product_stores)
            })
    return items

In [2]:
links = []
for i in range(5):
    search_url = f'https://www.babylist.com/find?search_term=Philadelphia&page={i}'
    search_page = get_page_html(search_url)
    hrefs = search_page.find_all('a', class_=re.compile(r"^results-container"))
    for a in hrefs:
        if a.get('href'):
            links.append(a.get('href'))
  

In [4]:
other_registeries= [
    'https://my.babylist.com/cqrxdkzuj?fbclid=IwZnRzaARN8ZFleHRuA2FlbQIxMQBzcnRjBmFwcF9pZAo2NjI4NTY4Mzc5AAEeYWQHPkagasJjYyHqCeJMu82tPU1oCK1qLkZbzFwRgljKQjD2I7SnZmq7pEI_aem_JX3jp0eReokL3T53O4cPtQ',
    'https://my.babylist.com/wmepkjyxf?fbclid=IwZnRzaARN8QlleHRuA2FlbQIxMQBzcnRjBmFwcF9pZAo2NjI4NTY4Mzc5AAEehXeaXwV4pO16UVK1GsN7tzc-aLeMwB50-H9qpvjR_N3WDUEJz09rC54j5_E_aem_fEYKunnoVKEz5uNegJOI1A',
]

registeries = np.unique(links + other_registeries)

items = []
for url in registeries:
    page = get_page_html(url)
    items.extend(extract_items(page))
    

In [5]:
df = pd.DataFrame(items)
df.to_csv('babylist_products.csv', index=False)
df

,listing,category,name,price,stores
0,Baby Caldwell's Baby Registry,Feeding,CATCHY - The Food Catcher for High Chairs – Ca...,$49.95,catchyandcrew
1,Baby Caldwell's Baby Registry,Feeding,MomMed S21 Double Wearable Breast Pump,$83.99,mommed
2,Baby Caldwell's Baby Registry,Feeding,Baby Food Feeder | Fresh Food & Fruit | Moonkie,$79.99,moonkieshop
3,Baby Caldwell's Baby Registry,Feeding,Small Single Pair - Silicone Flanges from Pump...,$24.95,pumpinpal
4,Baby Caldwell's Baby Registry,Sleeping,Throw Blanket,$69.00,babymorganblankets
...,...,...,...,...,...
20480,Meghan's Baby Registry,Purchased,Sassy Hello Baby PLAYMAT NR,N/A,
20481,Meghan's Baby Registry,Purchased,Itzy Ritzy Spiral Car Seat & Stroller Activity...,N/A,
20482,Meghan's Baby Registry,Purchased,Sassy Smart Stages Soft Book Set,N/A,
20483,Meghan's Baby Registry,Purchased,Sassy Sushi Sorter,N/A,


In [42]:
BRANDS = [
    "Philips Avent",
    "Pampers",
    "Huggies",
    "Coterie",
    "Millie Moon",
    "Dr. Brown's",
    "Comotomo",
    "Tommee Tippee",
    "MAM",
    "NUK",
    "Lansinoh",
    "Medela",
    "Haakaa",
    "Willow",
    "Elvie",
    "Kiinde",
    "Momcozy",
    "Boon",
    "OXO Tot",
    "Munchkin",
    "Skip Hop",
    "Frida Baby",
    "Frida Mom",
    "Safety 1st",
    "Dreambaby",
    "Ubbi",
    "Diaper Genie",
    "Boudreaux's",
    "Desitin",
    "Aquaphor",
    "Mustela",
    "Tubby Todd",
    "Aveeno Baby",
    "The Honest Company",
    "WaterWipes",
    "Babylist",
    "Target",
    "Amazon",
    "Graco",
    "Chicco",
    "Evenflo",
    "Nuna",
    "Doona",
    "UPPAbaby",
    "BabyBjörn",
    "Ergobaby",
    "Infantino",
    "4moms",
    "Ingenuity",
    "Fisher-Price",
    "Tiny Love",
    "Lovevery",
    "Bright Starts",
    "Baby Einstein",
    "Newton Baby",
    "Halo",
    "Hatch",
    "Owlet",
    "Nanit",
    "Babysense",
    "VTech",
    "Motorola",
    "DaVinci",
    "Storkcraft",
    "Babyletto",
    "Cloud Island",
    "Little Unicorn",
    "KeaBabies",
    "Comfy Cubs",
    "Copper Pearl",
    "Kyte Baby",
    "Burt's Bees Baby",
    "Carter's",
    "Gerber",
    "Zutano",
    "Magnetic Me",
    "Hanna Andersson",
    "Kindred Bravely",
    "Boppy",
    "My Brest Friend",
    "Itzy Ritzy",
    "Lalo",
    "AEIOU",
    "PandaEar",
    "Melissa & Doug",
    "Hape",
    "Lamaze",
    "Radio Flyer",
    "Skip Hop",
    "Sassy",
]
CATEGORY_MAP = {
    "Other Baby Registries": "Other",
    "Cash & Gift Certificates": "Other",
    "General": "Other",
    "Baby Gear": "Baby Gear",
    "Transportation": "Baby Gear",
    "Clothes & Accessories": "Clothing",
    "Feeding": "Feeding",
    "Bath Time": "Bathing",
    "Diapering": "Diapering",
    "Health & Safety": "Health & Safety",
    "Sleeping": "Sleeping",
    "Nursery & decor": "Nursery & decor",
    "Playing": "Playing",
    "Cash & gift cards": "Other",
    "Bathing": "Bathing",
    "Clothing": "Clothing",
    "For the parent(s)": "Mommy",
    "Help & favors": "Other",
    "Purchased": "Other",
    "Other": "Other",
    "Mommy": "Mommy",
    "Storage": "Nursery & decor",
    "Baby's library": "Playing",
    'Baby gear': "Baby Gear",
    'Health & safety': "Health & Safety",
    'Other baby registries': "Other",
    'Baby’s library': "Playing",
    'Toys':'Playing',
    'For mom': "Mommy",
    'Nursery & Decor': "Nursery & decor",
    'Toys & Books': "Playing",
    'For the Parent(s)': "Mommy",
    'Cash & Gift Cards': "Other",
    'For Mom': "Mommy",
    'DIAPER RAFFLE': "Diapering",
    'stroller': "Baby Gear",
    'Toiletries': "Health & Safety",
    'Baby bibs': "Feeding",
    'Pacifiers': "Feeding",
    'Must Haves': "Other",
    'Cash Funds': "Other",
    'Diapering essentials': "Diapering",
    'Swaddles and blankets': "Other",
    'Books': "Playing",
    'Travel': "Other",
    'Laundry': "Other",
    'Postpartum Supplies': "Health & Safety",
    'Mama Bear Aftercare': "Health & Safety",
    'Nursery🌷': "Nursery & decor",
    'Health 🤍': "Health & Safety",
    'Gear & Travel 👶🏼': "Baby Gear",
    'Clothing 🎀': "Clothing",
    'Bathing 🛁': "Bathing",
    'Diapering 💩': "Diapering",
    'Feeding 🍼': "Feeding",
    'Sleep 💤': "Sleeping",
    'Postpartum': "Health & Safety",
    'Books for Baby': "Playing",
    'Hospital': "Health & Safety",
    'Outerwear': "Clothing"
}

CATEGORY_SUBCATEGORY_MAP = {
    "Feeding": {
        "Bottle": ["bottle"],
        "Breast Pump": ["pump"],
        "Milk Storage": ["milk", "storage", "bag"],
        "Bib": ["bib"],
        "Burp Cloth": ["burp"],
        "Utensils": ["spoon", "fork", "plate"],
        "Sterilizer & Cleaning": ["sterilizer", "brush", "dryer"],
        "Warmer": ["warmer"],
        "High Chair": ["chair"],
    },
    "Diapering": {
        "Diapers": ["diaper"],
        "Wipes": ["wipe"],
        "Diaper Pail": ["pail", "genie"],
        "Changing Pad": ["changing", "pad"],
        "Diaper Bag": ["bag", "backpack"],
        "Cream": ["cream", "ointment", "rash"],
        "Organizer": ["caddy", "organizer"],
    },
    "Sleeping": {
        "Crib": ["crib"],
        "Bassinet": ["bassinet"],
        "Mattress": ["mattress"],
        "Sheets": ["sheet"],
        "Swaddle": ["swaddle", "sleep sack"],
        "Sound Machine": ["sound", "noise"],
        "Monitor": ["monitor"],
        "Humidifier": ["humidifier"],
    },
    "Bathing": {
        "Bathtub": ["tub", "bath"],
        "Towel": ["towel"],
        "Washcloth": ["washcloth"],
        "Skincare": ["lotion", "wash", "soap"],
        "Thermometer": ["thermometer"],
    },
    "Health & Safety": {
        "Thermometer": ["thermometer"],
        "Nasal Care": ["aspirator"],
        "Grooming Kit": ["groom", "nail"],
        "Safety Gear": ["gate", "lock", "proof"],
        "Medicine": ["tylenol", "medicine"],
    },
    "Baby Gear": {
        "Car Seat": ["car seat"],
        "Stroller": ["stroller"],
        "Travel System": ["travel system"],
        "Carrier": ["carrier"],
        "Accessories": ["cover", "mirror", "organizer"],
    },
    "Playing": {
        "Activity Gym": ["gym"],
        "Toy": ["toy", "ball", "rattle"],
        "Playmat": ["mat"],
        "Playpen": ["playpen"],
        "Walker": ["walker"],
        "Swing": ["swing", "bouncer"],
        "Books": ["book"],
    },
    "Clothing": {
        "Bodysuit": ["bodysuit", "onesie"],
        "Sleepwear": ["sleep", "pajama"],
        "Outerwear": ["jacket", "coat"],
        "Accessories": ["hat", "sock", "mitten"],
    },
    "Mommy": {
        "Postpartum": ["postpartum"],
        "Nursing": ["nursing", "bra", "pad"],
        "Recovery": ["recovery"],
    },
}


def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return text

normalize_category = {}
for key, val in CATEGORY_MAP.items():
    normalized_cat = normalize(key)
    normalize_category[normalized_cat] = normalize(val)
CATEGORY_MAP = normalize_category

normalized_subcategory_map = {}
for key, val in CATEGORY_SUBCATEGORY_MAP.items():
    normalized_cat = normalize(key)
    normalized_subcategory_map[normalized_cat] = val
CATEGORY_SUBCATEGORY_MAP = normalized_subcategory_map


def extract_brand(product_name):
    name = normalize(product_name)
    for brand in BRANDS:
        brand_clean = normalize(brand)

        # exact or partial match
        if brand_clean in name:
            return brand

    return "Generic"


def clean_category(cat):
    normalized_cat = normalize(cat)
    val = CATEGORY_MAP.get(normalized_cat, normalized_cat)

    return normalize(val)


def assign_subcategory(row):
    category = clean_category(row["category"])
    name = normalize(row["name"])

    if category not in CATEGORY_SUBCATEGORY_MAP:
        return "Other"

    sub_map = CATEGORY_SUBCATEGORY_MAP[category]

    scores = {}

    for subcategory, keywords in sub_map.items():
        score = sum(1 for kw in keywords if kw in name)
        if score > 0:
            scores[subcategory] = score

    if scores:
        return max(scores, key=scores.get)

    return "Other"


def get_cost_numeric(price):
    # the price is expected to be in a format like "$19.99" or "N/A"
    # or in a range like "$19.99-$29.99" take the mean of the range if it's a range
    if price == "N/A":
        return None
    if " - " in price:
        prices = price.split("-")
        lower = re.sub(r"[^\d.]", "", prices[0])
        upper = re.sub(r"[^\d.]", "", prices[1])
        try:
            lower_cost = float(lower)
            upper_cost = float(upper)
            return (lower_cost + upper_cost) / 2
        except ValueError:
            pass
    price = re.sub(r"[^\d.]", "", price)
    try:
        return float(price)
    except ValueError:
        return None

In [43]:
df["main_category"] = df["category"].apply(clean_category)
df["brand"] = df["name"].apply(extract_brand)
df["sub_category"] = df.apply(assign_subcategory, axis=1)
df["price_numeric"] = df["price"].apply(get_cost_numeric)
df

,listing,category,name,price,stores,main_category,brand,sub_category,price_numeric
0,Baby Caldwell's Baby Registry,Feeding,CATCHY - The Food Catcher for High Chairs – Ca...,$49.95,catchyandcrew,feeding,Generic,High Chair,49.95
1,Baby Caldwell's Baby Registry,Feeding,MomMed S21 Double Wearable Breast Pump,$83.99,mommed,feeding,Generic,Breast Pump,83.99
2,Baby Caldwell's Baby Registry,Feeding,Baby Food Feeder | Fresh Food & Fruit | Moonkie,$79.99,moonkieshop,feeding,Generic,Other,79.99
3,Baby Caldwell's Baby Registry,Feeding,Small Single Pair - Silicone Flanges from Pump...,$24.95,pumpinpal,feeding,Generic,Breast Pump,24.95
4,Baby Caldwell's Baby Registry,Sleeping,Throw Blanket,$69.00,babymorganblankets,sleeping,Generic,Other,69.00
...,...,...,...,...,...,...,...,...,...
20480,Meghan's Baby Registry,Purchased,Sassy Hello Baby PLAYMAT NR,N/A,,other,Sassy,Other,NaN
20481,Meghan's Baby Registry,Purchased,Itzy Ritzy Spiral Car Seat & Stroller Activity...,N/A,,other,Itzy Ritzy,Other,NaN
20482,Meghan's Baby Registry,Purchased,Sassy Smart Stages Soft Book Set,N/A,,other,Sassy,Other,NaN
20483,Meghan's Baby Registry,Purchased,Sassy Sushi Sorter,N/A,,other,Sassy,Other,NaN


In [44]:
df.to_csv('babylist_products.csv', index=False)

In [45]:
summary = df.groupby(["main_category", "sub_category"]).agg(
    count=("name", "size"),
    notes=("name", lambda x: '. '.join(x.unique() ))
).reset_index()

n = np.where(summary['sub_category']=='Other', 0, summary['count'])
summary['count'] = n
summary['group_total'] = summary.groupby("main_category")["count"].transform("sum")
summary['percentage'] = summary['count'] / (summary['group_total']+0.0001) * 100
summary['percentage_of_total'] = summary['count'] / summary['count'].sum() * 100
summary['priority'] = np.where(
    (summary['percentage'] > 25) | (summary['percentage_of_total'] > 1.5), 
    'Must Have', 
    np.where(
      (summary['percentage'] > 10) | (summary['percentage_of_total'] > 0.9),
    'Nice to Have',
    'Optional')
)

brand_summary = (
    df[df['brand']!='Generic'].groupby(["main_category", "sub_category", "brand"])
      .size()
      .reset_index(name="count")
)

def get_top_brands(group):
    group = group.sort_values("count", ascending=False)
    
    top_brand = group.iloc[0]["brand"] if len(group) > 0 else None
    alt_brand = group.iloc[1]["brand"] if len(group) > 1 else None
    
    return pd.Series({
        "top_brand": top_brand,
        "alternative_brand": alt_brand
    })

brand_summary = (
    brand_summary.groupby(["main_category", "sub_category"])
           .apply(get_top_brands)
           .reset_index()
)
priority_summary = summary[["main_category", "sub_category", "count", "priority", "notes"]].copy()

# Clean 'notes' by removing special characters and newlines
priority_summary['notes'] = priority_summary['notes'].apply(lambda x: re.sub(r'[^a-zA-Z0-9. ]', ' ', str(x)))

# Optional: Collapse multiple spaces into one for cleaner output
priority_summary['notes'] = priority_summary['notes'].str.replace(r'\s+', ' ', regex=True)


priority_summary = priority_summary.merge(brand_summary, on=["main_category", "sub_category"], how="left")
priority_summary

,main_category,sub_category,count,priority,notes,top_brand,alternative_brand
0,baby gear,Accessories,152,Nice to Have,MairMore Muslin Baby Cover for Newborn Soft an...,Boppy,Munchkin
1,baby gear,Car Seat,341,Must Have,EOS 5 in 1 Travel System Stroller Lightweight ...,Graco,Evenflo
2,baby gear,Carrier,238,Must Have,Thule Sapling Child Hiking Backpack Carrier Ag...,Momcozy,BabyBjörn
3,baby gear,Other,0,Optional,Carina. BabyBj rn Bouncer Bliss Sand Gray Dark...,Graco,Ingenuity
4,baby gear,Stroller,211,Must Have,UPPAbaby Vista V3 Stroller Aria Travel System ...,Momcozy,Nuna
...,...,...,...,...,...,...,...
57,sleeping,Monitor,146,Nice to Have,Bebcare Hear Low EMF Emissions Audio Baby Moni...,Owlet,Nanit
58,sleeping,Other,0,Optional,Throw Blanket. Old Navy x Disney Blanket for B...,Halo,Kyte Baby
59,sleeping,Sheets,13,Optional,Muslin Sheet for For Stokke Sleepi Mini Oval C...,Halo,Cloud Island
60,sleeping,Sound Machine,143,Nice to Have,Hatch Hatch Go Sound Machine Putty. Portable B...,Hatch,Momcozy


In [46]:
# save it to excel
priority_summary.to_excel('babylist_priority_summary.xlsx', index=False)